# 09 — Demo en vivo: clasificar un chip con los 4 modelos

Para la presentación: el profesor elige **cualquier PNG** del dataset (128×128).
Cambia `IMAGE_PATH` en la celda 2 y ejecuta todas las celdas. La celda 4 compara **los 4 modelos** `v2_bloques_tuned`.

**Requisitos previos:**
- Entorno activado: `pip install -e conf/`
- Los 4 `.pt` en `data/06_models/v2_bloques_tuned/` (ver README de esa carpeta)
- Dataset en: `data/01_raw/dataset_amazonia_garimpo_binario/`

Terminal (un modelo, por defecto ResNet-50):
`garimpo predict-image ruta/al/chip.png --checkpoint data/06_models/v2_bloques_tuned/swin_tiny_patch4_window7_224_best.pt`

## 1. Configuración

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

from src.config import MODELS_DIR, resolve_chips_dir
from src.models.predict import predict_image
from src.utils.helpers import get_device


CHIPS_DIR = resolve_chips_dir(PROJECT_ROOT / "data")
device = get_device()

CHECKPOINTS = sorted(MODELS_DIR.glob("*_best.pt"))

print(f"Proyecto    : {PROJECT_ROOT}")
print(f"Chips       : {CHIPS_DIR}")
print(f"Device      : {device}")
print(f"Checkpoints : {len(CHECKPOINTS)} en {MODELS_DIR}")
for p in CHECKPOINTS:
    print(f"  - {p.name}")

if len(CHECKPOINTS) < 4:
    raise FileNotFoundError(
        f"Se esperaban 4 .pt en {MODELS_DIR}. Copia todos según README de esa carpeta."
    )

## 2. Elegir imagen (cambiar solo esta celda en la demo)

Pega la ruta relativa desde `dataset_amazonia_garimpo_binario/` **o** la ruta absoluta al PNG.

Ejemplos:
- `com_garimpo/archivo.png`
- `sem_garimpo/archivo.png`

In [ ]:
# <<< CAMBIAR AQUÍ EN LA PRESENTACIÓN >>>
IMAGE_PATH = CHIPS_DIR / "com_garimpo" / "PONER_AQUI_EL_ARCHIVO.png"

# Si el profesor da solo el nombre relativo (com_garimpo/...):
# IMAGE_PATH = CHIPS_DIR / "com_garimpo/nombre_del_chip.png"

IMAGE_PATH = Path(IMAGE_PATH)
if not IMAGE_PATH.is_absolute():
    IMAGE_PATH = CHIPS_DIR / IMAGE_PATH

if not IMAGE_PATH.exists():
    raise FileNotFoundError(f"No existe: {IMAGE_PATH}")

print(IMAGE_PATH)

## 3. Ver el chip

In [ ]:
with Image.open(IMAGE_PATH) as img:
    chip = img.convert("RGB")

plt.figure(figsize=(4, 4))
plt.imshow(chip)
plt.axis("off")
plt.title(IMAGE_PATH.name, fontsize=9)
plt.show()

## 4. Predicción — los 4 modelos

In [ ]:
label_es = {
    "com_garimpo": "CON minería (garimpo)",
    "sem_garimpo": "SIN minería (selva)",
}

rows = []
for ckpt in CHECKPOINTS:
    r = predict_image(IMAGE_PATH, checkpoint_path=ckpt, device=device)
    rows.append({
        "modelo": r["model_name"],
        "checkpoint": ckpt.name,
        "predicción": label_es[r["label_name"]],
        "clase": r["label_name"],
        "P(garimpo)": round(r["prob_com_garimpo"], 4),
        "P(selva)": round(r["prob_sem_garimpo"], 4),
    })

comparativa = pd.DataFrame(rows)

print("=" * 60)
print(f"Imagen: {IMAGE_PATH.name}")
name = IMAGE_PATH.name.lower()
if "com_garimpo" in name:
    print("Referencia (filename): com_garimpo — con minería")
elif "sem_garimpo" in name:
    print("Referencia (filename): sem_garimpo — sin minería")
print("=" * 60)
display(comparativa)

# Modelo de producción (ResNet-50) resaltado en consola
prod = comparativa[comparativa["modelo"] == "resnet50"].iloc[0]
print(f"\nProducción (ResNet-50): {prod['predicción']} | P(garimpo)={prod['P(garimpo)']:.1%}")